# Lab 0 Task 0.1 — CIFAR-10 CNN Experiments
This notebook implements a simple CNN for CIFAR-10 classification and compares different optimizers and activation functions as required for Task 0.1 of Lab 0.

## Setup & Imports

In [1]:
!nvidia-smi # Take a look at the GPU

Sat Mar 28 11:49:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 2080 Ti     On  |   00000000:1A:00.0 Off |                  N/A |
| 29%   25C    P8              1W /  250W |    3786MiB /  11264MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import warnings
import wandb
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10
from typing import List, Dict, Tuple, Any
from utils import make_loaders, train, validate, fit, evaluate

# Suppress NumPy 2.4 VisibleDeprecationWarning triggered inside torchvision
warnings.filterwarnings("ignore", category=np.exceptions.VisibleDeprecationWarning)

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Make results reproducible
torch.manual_seed(1)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(1)

2.11.0+cu130
True
NVIDIA GeForce RTX 2080 Ti
Using device: cuda


## Notebook parameters

In [3]:
LOG_WANDB = True # Notebook level parameter for enabling/disabling wandb logging
NUM_WORKERS = 8
PIN_MEMORY = True

## Data Loading

In [4]:
BATCH_SIZE = 256
VAL_FRACTION = 0.2  # 20% of training set used for validation

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

train_set = CIFAR10(root="../data", train=True,  download=True, transform=transform)
test_set  = CIFAR10(root="../data", train=False, download=True, transform=transform)

loader_kwargs = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

train_loader, val_loader, test_loader = make_loaders(
    train_set, test_set,
    loader_kwargs=loader_kwargs,
    val_fraction=VAL_FRACTION,
)

classes: list[str] = train_set.classes
print("Classes:", classes)

Dataset split — train: 40,000  val: 10,000  test: 10,000
Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


## Model Definition

The `activation` argument lets us swap between `LeakyReLU` and `Tanh` without rewriting the class.

In [5]:
class SimpleCNN(nn.Module):
    """
    A simple CNN for CIFAR-10 classification.

    Architecture:
        Conv Block 1 : Conv2d(3  → 32)  → act → MaxPool2d  (32×32 → 16×16)
        Conv Block 2 : Conv2d(32 → 64)  → act → MaxPool2d  (16×16 →  8×8)
        Conv Block 3 : Conv2d(64 → 128) → act → MaxPool2d  ( 8×8  →  4×4)
        FC 1         : Linear(128*4*4 → 256) → act
        FC 2         : Linear(256 → num_classes)
    """

    def __init__(self, num_classes: int = 10, activation: nn.Module = nn.LeakyReLU()) -> None:
        super().__init__()

        self.conv1 = nn.Conv2d(3,  32,  kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64,  kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)

        self.act  = activation
        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.pool(self.act(self.conv1(x)))
        x = self.pool(self.act(self.conv2(x)))
        x = self.pool(self.act(self.conv3(x)))
        x = torch.flatten(x, start_dim=1)
        x = self.act(self.fc1(x))
        return self.fc2(x)

## Experiment A – SGD + LeakyReLU
*(The original baseline from the script)*

In [6]:
# Hyperparameters
NUM_EPOCHS = 50
LEARNING_RATE = 1e-2

model_a = SimpleCNN(num_classes=len(classes), activation=nn.LeakyReLU()).to(device)
optimizer_a = optim.SGD(model_a.parameters(), lr=LEARNING_RATE)

wandb_kwargs = dict(
    entity="d7047e-group12",
    project="Lab0",
    name="SGD + LeakyReLU",
    tags=["Task 0.1"],
    config={"optimizer": "SGD", "lr": LEARNING_RATE, "activation": "LeakyReLU",
            "epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE},
)

history_a = fit(
    model=model_a, 
    optimizer=optimizer_a,
    criterion=nn.CrossEntropyLoss(),
    train_loader=train_loader,
    val_loader=val_loader,
    wandb_kwargs=wandb_kwargs,
    num_epochs=NUM_EPOCHS,
    log=LOG_WANDB
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: oscar-engelmark (d7047e-group12) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch  1/50 | train loss 2.2984, train acc 13.96% | val loss 2.2937, val acc 18.09%
Epoch  2/50 | train loss 2.2865, train acc 16.89% | val loss 2.2765, val acc 18.56%
Epoch  3/50 | train loss 2.2535, train acc 20.69% | val loss 2.2164, val acc 22.96%
Epoch  4/50 | train loss 2.1565, train acc 23.34% | val loss 2.0957, val acc 25.53%
Epoch  5/50 | train loss 2.0504, train acc 26.82% | val loss 2.0069, val acc 29.34%
Epoch  6/50 | train loss 1.9575, train acc 30.66% | val loss 1.9097, val acc 31.90%
Epoch  7/50 | train loss 1.8661, train acc 33.48% | val loss 1.8406, val acc 33.22%
Epoch  8/50 | train loss 1.7980, train acc 35.69% | val loss 1.7860, val acc 34.64%
Epoch  9/50 | train loss 1.7431, train acc 37.23% | val loss 1.7771, val acc 34.68%
Epoch 10/50 | train loss 1.7054, train acc 38.80% | val loss 1.7329, val acc 37.09%
Epoch 11/50 | train loss 1.6607, train acc 40.47% | val loss 1.6663, val acc 39.66%
Epoch 12/50 | train loss 1.6233, train acc 41.60% | val loss 1.6471, val acc

Training Accuracy,▁▁▂▂▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇██████
Training Loss,███▇▇▆▅▅▅▅▄▄▄▄▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁
Validation Accuracy,▁▁▂▂▃▄▄▄▅▅▅▅▅▅▆▅▆▆▆▆▇▆▇▆▇▇▆▇▇▇▇▇▇██▇██▇█
Validation Loss,███▇▆▅▅▅▄▄▄▃▄▃▃▃▃▃▃▄▃▂▂▂▂▃▂▂▂▁▂▂▂▁▁▂▁▁▂▁
Training Accuracy,66.5475
Training Loss,0.96414
Validation Accuracy,58.95
Validation Loss,1.16927



Restored best weights (val acc 62.23%)


In [7]:
_ = evaluate(model=model_a, test_loader=test_loader,
         criterion=nn.CrossEntropyLoss(), label="SGD + LeakyReLU")

[SGD + LeakyReLU] Test loss: 1.0587 | Test acc: 63.22%


## Experiment B – Adam + LeakyReLU
*(Task: swap SGD for Adam and report accuracy)*

In [8]:
# Hyperparameters
NUM_EPOCHS = 50
LEARNING_RATE = 1e-3

model_b     = SimpleCNN(num_classes=len(classes), activation=nn.LeakyReLU()).to(device)
optimizer_b = optim.Adam(model_b.parameters(), lr=LEARNING_RATE)

wandb_kwargs = dict(
    entity="d7047e-group12",
    project="Lab0",
    name="Adam + LeakyReLU",
    tags=["Task 0.1"],
    config={"optimizer": "Adam", "lr": LEARNING_RATE, "activation": "LeakyReLU",
            "epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE},
)

history_b = fit(
    model=model_b, 
    optimizer=optimizer_b,
    criterion=nn.CrossEntropyLoss(),
    train_loader=train_loader,
    val_loader=val_loader,
    wandb_kwargs=wandb_kwargs,
    num_epochs=NUM_EPOCHS,
    log=LOG_WANDB
)

Epoch  1/50 | train loss 1.6006, train acc 41.64% | val loss 1.3406, val acc 51.58%
Epoch  2/50 | train loss 1.2043, train acc 56.99% | val loss 1.1495, val acc 59.08%
Epoch  3/50 | train loss 1.0007, train acc 64.28% | val loss 0.9763, val acc 65.72%
Epoch  4/50 | train loss 0.8678, train acc 69.39% | val loss 0.9661, val acc 65.91%
Epoch  5/50 | train loss 0.7722, train acc 72.94% | val loss 0.8671, val acc 69.65%
Epoch  6/50 | train loss 0.6833, train acc 76.25% | val loss 0.8486, val acc 70.60%
Epoch  7/50 | train loss 0.6002, train acc 79.32% | val loss 0.8534, val acc 70.93%
Epoch  8/50 | train loss 0.5324, train acc 81.34% | val loss 0.7901, val acc 73.53%
Epoch  9/50 | train loss 0.4477, train acc 84.48% | val loss 0.8131, val acc 73.80%
Epoch 10/50 | train loss 0.3762, train acc 87.06% | val loss 0.9302, val acc 71.20%
Epoch 11/50 | train loss 0.3188, train acc 88.94% | val loss 0.8433, val acc 73.49%
Epoch 12/50 | train loss 0.2507, train acc 91.50% | val loss 0.9876, val acc

Training Accuracy,▁▃▄▄▅▆▆▆▆▇▇▇████████████████████████████
Training Loss,█▆▅▅▄▄▃▃▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Validation Accuracy,▁▃▅▇▇██▇█▇███▇███▇█▇██▇████▇▇███▇█▇█▇███
Validation Loss,▃▃▂▂▁▁▁▂▁▂▂▂▃▄▄▄▄▅▅▅▅▆▆▆▅▆▆▆▇▆▇▇▇▇▇██▇▇█
Training Accuracy,99.3025
Training Loss,0.02066
Validation Accuracy,72.95
Validation Loss,2.31519



Restored best weights (val acc 74.26%)


In [9]:
_ = evaluate(model=model_b, test_loader=test_loader,
         criterion=nn.CrossEntropyLoss(), label="Adam + LeakyReLU")

[Adam + LeakyReLU] Test loss: 1.9739 | Test acc: 73.84%


## Experiment C – Adam + Tanh
*(Task: swap LeakyReLU for Tanh and report accuracy)*

In [10]:
# Hyperparameters
NUM_EPOCHS = 50
LEARNING_RATE = 1e-3

model_c     = SimpleCNN(num_classes=len(classes), activation=nn.Tanh()).to(device)
optimizer_c = optim.Adam(model_c.parameters(), lr=LEARNING_RATE)

wandb_kwargs = dict(
    entity="d7047e-group12",
    project="Lab0",
    name="Adam + Tanh",
    tags=["Task 0.1"],
    config={"optimizer": "Adam", "lr": LEARNING_RATE, "activation": "Tanh",
            "epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE},
)

history_c = fit(
    model=model_c, 
    optimizer=optimizer_c,
    criterion=nn.CrossEntropyLoss(),
    train_loader=train_loader,
    val_loader=val_loader,
    wandb_kwargs=wandb_kwargs,
    num_epochs=NUM_EPOCHS,
    log=LOG_WANDB
)

Epoch  1/50 | train loss 1.4624, train acc 47.69% | val loss 1.2723, val acc 54.01%
Epoch  2/50 | train loss 1.0718, train acc 62.12% | val loss 0.9984, val acc 64.64%
Epoch  3/50 | train loss 0.8845, train acc 68.84% | val loss 0.9520, val acc 66.53%
Epoch  4/50 | train loss 0.7733, train acc 73.11% | val loss 0.8552, val acc 70.34%
Epoch  5/50 | train loss 0.6590, train acc 77.34% | val loss 0.8652, val acc 69.92%
Epoch  6/50 | train loss 0.5630, train acc 80.89% | val loss 0.8212, val acc 72.11%
Epoch  7/50 | train loss 0.4682, train acc 84.56% | val loss 0.8256, val acc 72.62%
Epoch  8/50 | train loss 0.3800, train acc 87.82% | val loss 0.8137, val acc 73.16%
Epoch  9/50 | train loss 0.2984, train acc 91.01% | val loss 0.8360, val acc 72.74%
Epoch 10/50 | train loss 0.2225, train acc 93.92% | val loss 0.8589, val acc 73.13%
Epoch 11/50 | train loss 0.1620, train acc 96.17% | val loss 0.8941, val acc 73.14%
Epoch 12/50 | train loss 0.1070, train acc 98.14% | val loss 0.9240, val acc

Training Accuracy,▁▃▄▄▅▆▆▇▇▇██████████████████████████████
Training Loss,█▆▅▅▄▃▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Validation Accuracy,▁▅▅▇▆▇█▇████████████████████████████████
Validation Loss,▆▃▂▁▁▁▁▁▂▂▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇███
Training Accuracy,100
Training Loss,0.00021
Validation Accuracy,73.86
Validation Loss,1.51427



Restored best weights (val acc 74.39%)


In [11]:
_ = evaluate(model=model_c, test_loader=test_loader,
         criterion=nn.CrossEntropyLoss(), label="Adam + Tanh")

[Adam + Tanh] Test loss: 1.1575 | Test acc: 74.04%
